In [ ]:
# ============================================================================
# PARAMETERS — this cell is identical in all three notebooks.
# ============================================================================
NOTEBOOK_NAME = "04_final_training_run"    # identity + build fingerprint of THESE cells;
NOTEBOOK_BUILD = "cdb44910ce5b"  # checked against the repo so stale cells fail loudly
RUN_MODE = "micro"          # "smoke" | "micro" (default) | "budget" | "full"
NUM_GPUS = None             # None = use every visible GPU; set 1 to force single-GPU
PUBLISH_KAGGLE_DATASET = True
CKPT_DATASET_SLUG = "dentex-repro-ckpts"
DATA_DATASET_SLUG = "dentex-repro-data"
REPO_URL = "https://github.com/christopherh-88/HierarchicalDet.git"

import os, subprocess, sys

# On Kaggle the repo is cloned into /kaggle/working (the only writable place
# that survives "Save Version"); locally the notebook already sits inside it.
if os.path.isdir("/kaggle/working"):
    CLONE = "/kaggle/working/repo"
    if os.path.isdir(os.path.join(CLONE, ".git")):
        subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "50", REPO_URL, CLONE], check=True)
    PROJECT_ROOT = os.path.join(CLONE, "dentex-repro")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["RUN_MODE"] = RUN_MODE
print("project root:", PROJECT_ROOT)


In [ ]:
# ---- Environment: install, pin, and prove the VENDORED code is what loaded ----
# Kaggle reverts to its base image every session, so this runs every time.
import json
from src import setup_env

# `git pull` above refreshed src/ and configs_repro/ -- but NOT these cells,
# which are the copy uploaded to Kaggle. Fail loudly rather than run a mix.
print("notebook build:", setup_env.assert_notebook_current(NOTEBOOK_NAME, NOTEBOOK_BUILD))
setup_env.install_dependencies()
# The vendored pycocotools ships Python sources only; its compiled `_mask`
# extension is grafted in here and VERIFIED BY IMPORT. It is compiled against
# numpy's C ABI, so a mismatch surfaces as "numpy.dtype size changed" deep
# inside detectron2.structures — which reads as a detectron2 problem and is not.
import numpy
print("numpy {} | pycocotools _mask -> {}".format(
    numpy.__version__, setup_env.ensure_pycocotools_mask()))
run = setup_env.bootstrap(RUN_MODE, require_gpu=True)

from src import manifest, train_utils

NUM_GPUS = NUM_GPUS if NUM_GPUS is not None else max(1, train_utils.visible_gpus())
lock = setup_env.write_requirements_lock()
environment = setup_env.env_report()
manifest.record_environment(environment)

# The repo vendors MODIFIED detectron2 / pycocotools (multi-label partial
# annotations, 3-tier category schema). A pip-installed copy silently shadows
# them and every number changes, so this is an assertion, not a warning.
found = setup_env.assert_vendored()
for module, path in found.items():
    print("{:14s} -> {}".format(module, path))
import detectron2, pycocotools, evaluator                              # noqa: F401
from hierarchialdet.util.coco_3class_eval import COCOEvaluator         # noqa: F401
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper  # noqa: F401
print("full import chain OK | commit {} | {} GPU process(es)".format(
    environment["repo_commit"][:12], NUM_GPUS))


In [ ]:
# ---- Inputs, weights, configs (identical to notebook 02) ----
from src import data_convert, degradations, eval_utils, registration

TRAIN_FOR_FULL_CLAIM_COVERAGE = False   # adds base DiffusionDet + every variant

paths = data_convert.layout()
for key in ("train_quadrant", "train_enumeration", "train_diagnosis", "val_diagnosis",
            "test_diagnosis"):
    assert os.path.exists(paths[key]), (
        "{} is missing - run notebook 01, or attach the {} dataset"
        .format(paths[key], DATA_DATASET_SLUG))

IMAGENET_WEIGHTS = train_utils.ensure_swin_weights()
CFG = {name: os.path.join(setup_env.CONFIGS_REPRO, "diffdet.dentex.{}.yaml".format(name))
       for name in ("quadrant", "enumeration", "diagnosis", "base_diffusiondet")}
NOISY_DIR = os.path.join(setup_env.RUNS_DIR, "noisy_boxes")
os.makedirs(NOISY_DIR, exist_ok=True)
print("ImageNet Swin-B:", IMAGENET_WEIGHTS)
print("GPU-hours already recorded on disk: {:.2f}".format(setup_env.gpu_hours_spent()))
print("full claim coverage:", TRAIN_FOR_FULL_CLAIM_COVERAGE)


In [ ]:
# ---- Does multi-GPU actually work in this Kaggle session? ----
# Copied from notebook 02, which learned this the expensive way: a probe on the
# diagnosis stage passed while the real quadrant run died at iteration 2. The
# probe must exercise the quadrant shape (unsupervised heads, the DDP worst
# case), and a failure falls back to one GPU instead of poisoning every stage.
import shutil

ddp = {"requested": NUM_GPUS, "works": None, "error": None}
probe_dir = os.path.join(setup_env.RUNS_DIR, "ddp_probe")
if NUM_GPUS > 1 and not os.path.exists(os.path.join(setup_env.RUNS_DIR, ".ddp_ok")):
    try:
        train_utils.launch_training(
            CFG["quadrant"],
            train_utils.base_overrides(run, probe_dir, 20, IMAGENET_WEIGHTS, NUM_GPUS),
            registration.training_env("quadrant_train"), probe_dir, NUM_GPUS,
            resume=False, log_name="ddp_probe.log")
        ddp["works"] = True
        open(os.path.join(setup_env.RUNS_DIR, ".ddp_ok"), "w").close()
    except RuntimeError as error:
        ddp["works"] = False
        ddp["error"] = str(error)[-1500:]
        NUM_GPUS = 1
        setup_env.log_deviation(
            "multi-GPU training disabled (DDP failed in this Kaggle session)",
            "launch(num_gpus=2) failed during the probe; the study runs single-GPU "
            "rather than fighting a flaky DDP setup", NOTEBOOK_NAME,
            impact="effective batch size halves relative to a 2-GPU run")
else:
    ddp["works"] = "already verified" if NUM_GPUS > 1 else "single GPU"
shutil.rmtree(probe_dir, ignore_errors=True)
print(json.dumps(ddp, indent=2), "-> NUM_GPUS =", NUM_GPUS)


In [ ]:
# ---- Pre-flight: a short real run + a real evaluation, on 10 images ----
# Proves the train -> checkpoint -> evaluate chain end to end BEFORE committing
# hours of quota to it. 30 iterations cannot detect anything; the assertion is
# that the pipeline ran, not that it scored.
PREFLIGHT_ITERS = 30
smoke = {"skipped": run.is_smoke}
if not run.is_smoke and not train_utils.is_complete("preflight"):
    smoke_dir = train_utils.run_dir("preflight")
    train_utils.launch_training(
        CFG["diagnosis"],
        train_utils.base_overrides(run, smoke_dir, PREFLIGHT_ITERS,
                                   IMAGENET_WEIGHTS, NUM_GPUS),
        registration.training_env("diagnosis_train"), smoke_dir, NUM_GPUS, resume=False)
    smoke_weights = os.path.join(smoke_dir, "model_final.pth")
    assert os.path.exists(smoke_weights), "pre-flight produced no model_final.pth"
    smoke_eval = eval_utils.evaluate_checkpoint(
        smoke_weights, CFG["diagnosis"], split="diagnosis_test", tier=2, seed=0, limit=10)
    assert set(smoke_eval["tiers"]) == {"quadrant", "enumeration", "diagnosis"}
    smoke = {"weights": smoke_weights,
             "metrics": {t: p["metrics"] for t, p in smoke_eval["tiers"].items()}}
    shutil.rmtree(smoke_dir, ignore_errors=True)
print(json.dumps(smoke, indent=2, default=str))


In [ ]:
# ---- Throughput calibration, then stage 0: quadrant ----
calibration = None
if run.quadrant.max_iter is None or run.diagnosis.max_iter is None:
    calibration = train_utils.calibrate_rate(
        "swinb_gpu{}".format(NUM_GPUS), CFG["quadrant"], run,
        "quadrant_train", IMAGENET_WEIGHTS, NUM_GPUS)
    print(json.dumps(calibration, indent=2))

training_records = []
quadrant_record = train_utils.train_stage(
    "quadrant_stage", CFG["quadrant"], run, "quadrant_train",
    IMAGENET_WEIGHTS, NUM_GPUS, run.quadrant, calibration=calibration)
quadrant_record["kind"] = "prerequisite"
training_records.append(quadrant_record)
QUADRANT_WEIGHTS = train_utils.final_weights("quadrant_stage")
print("quadrant: {} iters, {} s".format(
    quadrant_record.get("max_iter"), quadrant_record.get("wall_seconds")))


In [ ]:
# ---- Stage 1: enumeration, seeded by the quadrant model's boxes ----
enum_boxes = {}
for key, split in (("NOISY_BOX_TRAIN", "quadrant_enumeration_train"),
                   ("NOISY_BOX_VAL", "diagnosis_val")):
    output = os.path.join(NOISY_DIR, "quadrant_over_{}.json".format(split))
    if not os.path.exists(output):
        eval_utils.dump_predictions(QUADRANT_WEIGHTS, CFG["quadrant"], split, 0,
                                    output, seed=0, limit=run.eval_limit)
    enum_boxes[key] = output

enumeration_record = train_utils.train_stage(
    "enumeration_stage", CFG["enumeration"], run, "quadrant_enumeration_train",
    QUADRANT_WEIGHTS, NUM_GPUS, run.enumeration, calibration=calibration,
    noisy_boxes=enum_boxes)
enumeration_record["kind"] = "prerequisite"
training_records.append(enumeration_record)
ENUM_WEIGHTS = train_utils.final_weights("enumeration_stage")
print("enumeration: {} iters, {} s".format(
    enumeration_record.get("max_iter"), enumeration_record.get("wall_seconds")))


In [ ]:
# ---- Enumeration model -> noisy boxes, and the prior-tier check ----
diagnosis_boxes = {}
for key, split in (("NOISY_BOX_TRAIN", "diagnosis_train"),
                   ("NOISY_BOX_VAL", "diagnosis_val")):
    output = os.path.join(NOISY_DIR, "enumeration_over_{}.json".format(split))
    if not os.path.exists(output):
        eval_utils.dump_predictions(ENUM_WEIGHTS, CFG["enumeration"], split, 1,
                                    output, seed=0, limit=run.eval_limit)
    diagnosis_boxes[key] = output

prior_test = os.path.join(NOISY_DIR, "enumeration_over_diagnosis_test.json")
if not os.path.exists(prior_test):
    eval_utils.dump_predictions(ENUM_WEIGHTS, CFG["enumeration"], "diagnosis_test", 1,
                                prior_test, seed=0, limit=run.eval_limit)

# The check that was missing. Notebook 03 injects these boxes to measure how
# sensitive the diagnosis tier is to an imperfect prior tier; the detector keeps
# only boxes scoring >= 0.5 and injects nothing at all if none qualify, in which
# case every severity silently evaluates the unperturbed model and the sweep
# reports six identical numbers. Catch it here, while the enumeration model is
# still the thing under discussion.
coverage = eval_utils.prior_box_coverage(prior_test)
print(json.dumps(coverage, indent=2))
if not coverage["boxes_kept"]:
    setup_env.log_deviation(
        "hierarchy fault injection unavailable: no prior-tier box scored >= 0.5",
        "the enumeration model produced {} test-split detections, none above the "
        "injection threshold, so inference-time injection is a no-op and the "
        "sweep cannot measure prior-tier sensitivity"
        .format(coverage["boxes_total"]), NOTEBOOK_NAME,
        impact="notebook 03 skips the fault-injection experiment instead of "
               "reporting the unperturbed model as its result")
    print("\nWARNING: fault injection will be SKIPPED downstream.")
box_stats = {key: degradations.summarize_prediction_file(path)
             for key, path in diagnosis_boxes.items()}
print(json.dumps(box_stats, indent=2))


In [ ]:
# ---- Stage 2: the diagnosis variants, at identical budgets ----
# Built directly from the switch table rather than through variant_plan():
# variant_plan expands run.variants only, and wo_manip_transfer is not in
# micro's list -- filtering its output could therefore never ADD the variant
# that full claim coverage exists to add.
variants = setup_env.ALL_VARIANTS if TRAIN_FOR_FULL_CLAIM_COVERAGE else run.variants
plans = []
for variant in variants:
    switches = setup_env.VARIANT_SWITCHES[variant]
    plans.append({
        "variant": variant,
        "label": setup_env.VARIANT_LABELS[variant],
        "run_name": "diagnosis_{}".format(variant),
        "weights": ENUM_WEIGHTS if switches["transfer"] else IMAGENET_WEIGHTS,
        "noisy_boxes": diagnosis_boxes if switches["manipulation"] else None,
        "switches": switches,
    })
variant_records = []
for plan in plans:
    print("\n" + "=" * 72)
    print("{}  (transfer={}, manipulation={})".format(
        plan["variant"], plan["switches"]["transfer"], plan["switches"]["manipulation"]))
    print("=" * 72)
    record = train_utils.train_stage(
        plan["run_name"], CFG["diagnosis"], run, "diagnosis_train",
        plan["weights"], NUM_GPUS, run.diagnosis, calibration=calibration,
        noisy_boxes=plan["noisy_boxes"],
        trajectory_fractions=run.trajectory_fractions)
    record.update({"variant": plan["variant"], "label": plan["label"],
                   "switches": plan["switches"], "kind": "diagnosis_variant"})
    variant_records.append(record)
    training_records.append(record)

# Different wall time between variants is fine; different iteration counts,
# seeds or batch sizes are not - that measures the budget, not the switch.
matched = train_utils.assert_matched_budgets(variant_records)
print("\n" + json.dumps(matched, indent=2, default=str))


In [ ]:
# ---- Base DiffusionDet, which claim 1 needs ----
TIER_CLASSES = {0: 4, 1: 8, 2: 4}
TIER_SPLIT = {0: "quadrant_train", 1: "quadrant_enumeration_train", 2: "diagnosis_train"}
base_tiers = (2,) if TRAIN_FOR_FULL_CLAIM_COVERAGE else run.base_tiers
base_records = []
for tier in base_tiers:
    flat_train = data_convert.flat_json_path(tier, "train")
    assert os.path.exists(flat_train), "run notebook 01 first: {}".format(flat_train)
    print("\n=== base_diffusiondet_tier{} ===".format(tier))
    record = train_utils.train_stage(
        "base_diffusiondet_tier{}".format(tier), CFG["base_diffusiondet"], run,
        TIER_SPLIT[tier], IMAGENET_WEIGHTS, NUM_GPUS, run.base_diffusiondet,
        extra_overrides=["MODEL.DiffusionDet.NUM_CLASSES",
                         "[{}, 8, 4]".format(TIER_CLASSES[tier])],
        env_override={"TRAIN_JSON": flat_train,
                      "VAL_JSON": data_convert.flat_json_path(tier, "test"),
                      "VAL_IMG_DIR": paths["img_test"], "TIER": "0"})
    record.update({"tier": tier, "flat_train_json": flat_train,
                   "kind": "base_diffusiondet"})
    base_records.append(record)
    training_records.append(record)
if not base_tiers:
    print("base DiffusionDet skipped; claim 1 stays 'cited, untested'")


In [ ]:
# ---- Retain the run records (the gap this notebook exists to close) ----
# runs/<mode>/<stage>/run_record.json is where the training cost lives.
# setup_env.gpu_hours_spent() walks exactly these files, and the checklist's
# Runs table and Exact commands come from them. Left in the session they vanish
# with it, and the study's compute becomes unverifiable - which is what
# happened: scope_and_claims.md reported 0.00 GPU-hours against a real spend.
import shutil

RECORDS_DIR = os.path.join(setup_env.RESULTS_RAW, "run_records")
os.makedirs(RECORDS_DIR, exist_ok=True)
retained = []
for root, _dirs, files in os.walk(setup_env.RUNS_ROOT):
    if "run_record.json" not in files:
        continue
    stage = os.path.basename(root)
    destination = os.path.join(RECORDS_DIR, "{}.json".format(stage))
    shutil.copy2(os.path.join(root, "run_record.json"), destination)
    retained.append(os.path.relpath(destination, setup_env.PAPER_ASSETS))
print("retained {} run record(s):".format(len(retained)))
for item in sorted(retained):
    print(" ", item)
print("\nGPU-hours these records account for: {:.2f}".format(
    setup_env.gpu_hours_spent()))


In [ ]:
# ---- Is this run strong enough to support a claim? ----
# An ablation ordering reproduced at a tenth of the paper's accuracy is not
# evidence about the mechanism: undertraining reproduces orderings too. The
# claim-coverage matrix downgrades below this floor, so state it here, next to
# the run that decides it.
from src import tables

DEGENERATE_AP = 10.0
reference = tables.reference_lookup()["diagnosis"]
verdict = {}
for record in variant_records:
    weights = train_utils.final_weights(record["name"])
    result = eval_utils.evaluate_checkpoint(
        weights, CFG["diagnosis"], split="diagnosis_test", tier=2, seed=0,
        limit=run.eval_limit)
    measured = result["tiers"]["diagnosis"]["metrics"].get("AP")
    expected = (reference.get(record["label"], {}) or {}).get("AP")
    verdict[record["label"]] = {
        "AP": measured, "paper_AP": expected,
        "fraction_of_paper": (measured / expected) if (measured and expected) else None,
        "degenerate": measured is not None and measured < DEGENERATE_AP,
    }
    print("{:26s} AP {:6.2f}   paper {:5.1f}   {}".format(
        record["label"], measured or 0.0, expected or 0.0,
        "DEGENERATE" if verdict[record["label"]]["degenerate"] else "ok"))
if any(v["degenerate"] for v in verdict.values()):
    print("\nAt least one arm is below AP {:.0f}. The ablation ordering cannot be "
          "separated from undertraining, and scope_and_claims.md will mark the "
          "manipulation claim 'undertrained, not conclusive'. Raise RUN_MODE or "
          "the iteration budget before treating this as a reproduction."
          .format(DEGENERATE_AP))


In [ ]:
# ---- Notebook summary (the only cross-notebook contract) ----
summary = {
    "run_mode": run.mode,
    "num_gpus": NUM_GPUS,
    "multi_gpu": ddp,
    "preflight": smoke,
    "calibration": calibration,
    "records": training_records,
    "variants_trained": [r["variant"] for r in variant_records],
    "matched_budgets": matched,
    "full_claim_coverage": TRAIN_FOR_FULL_CLAIM_COVERAGE,
    "weights": {
        "imagenet": IMAGENET_WEIGHTS,
        "quadrant": QUADRANT_WEIGHTS,
        "enumeration": ENUM_WEIGHTS,
        "variants": {r["variant"]: train_utils.final_weights(r["name"])
                     for r in variant_records},
        "base": {str(r["tier"]): train_utils.final_weights(r["name"])
                 for r in base_records},
    },
    "noisy_boxes": {"for_enumeration": enum_boxes, "for_diagnosis": diagnosis_boxes,
                    "prior_over_test": prior_test, "stats": box_stats},
    "prior_box_coverage": coverage,
    "trajectory_checkpoints": {r["variant"]: r.get("trajectory", {})
                               for r in variant_records},
    "degeneracy": verdict,
    "retained_run_records": sorted(retained),
    "gpu_hours_spent": setup_env.gpu_hours_spent(),
}

path = setup_env.write_notebook_summary("04_final_training_run", summary)
print("wrote", path)
print(json.dumps(summary, indent=2, default=str)[:4000])


In [ ]:
# ---- Publish, last ----
# paper_assets is included so the retained run records and this summary travel
# with the checkpoints; notebook 03 reads both.
publish = {"status": "disabled"}
if PUBLISH_KAGGLE_DATASET:
    publish = train_utils.publish_kaggle_dataset(
        CKPT_DATASET_SLUG, [setup_env.RUNS_DIR, setup_env.PAPER_ASSETS],
        "final training run ({} mode)".format(run.mode))
    summary["kaggle_publish"] = {k: v for k, v in publish.items()
                                 if k not in ("stdout", "stderr")}
    setup_env.write_notebook_summary("04_final_training_run", summary)
print(json.dumps({k: v for k, v in publish.items() if k not in ("stdout", "stderr")},
                 indent=2))
print("\nGPU-hours recorded: {:.2f}".format(setup_env.gpu_hours_spent()))
print("Attach to notebook 03 as: {} (plus {})".format(
    CKPT_DATASET_SLUG, DATA_DATASET_SLUG))
